# 📱 GroupDNA – WhatsApp Group Chat Analyzer

### The Unlox Academy – Minor Project

**Name:** Poorvika

**Internship:** Unlox Academy

**Technology:** Python + NumPy

---

## 📑 Table of Contents

1. Import Libraries
2. Read Dataset
3. Chat Parser
4. Group Overview
5. Most Active Day & Hour
6. NumPy Activity Heatmap
7. Top 10 Words
8. Response Analysis
9. Personality Detection
10. Final Report

In [ ]:
# ==========================================================
# GROUPDNA - WhatsApp Group Chat Analyzer
# Import Required Libraries
# ==========================================================

import numpy as np
from datetime import datetime

print("=" * 60)
print("GROUPDNA - WhatsApp Group Chat Analyzer")
print("Libraries Imported Successfully")
print("=" * 60)

GROUPDNA - WhatsApp Group Chat Analyzer
Libraries Imported Successfully


# Feature 1 – Reading the Dataset

In [ ]:
# ==========================================================
# Reading the Dataset
# ==========================================================

file_name = "DADS Minor PROJECT dataset.txt"

with open(file_name, "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Dataset Loaded Successfully!")
print("Total Lines:", len(lines))

print("\nFirst 5 Lines:\n")

for line in lines[:5]:
    print(line.strip())

Dataset Loaded Successfully!
Total Lines: 3178

First 5 Lines:

01/04/24, 01:12 - Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them. Tap to learn more.
01/04/24, 01:13 - Priya created group "Hostel Bois 4ever"
01/04/24, 01:14 - Priya added you
01/04/24, 01:17 - Rahul: scene fix
01/04/24, 01:17 - Rahul: haan


# Feature 2 – Chat Parser

In [ ]:
# ==========================================================
# Feature 2 : Chat Parser
# ==========================================================

messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0

for line in lines:

    line = line.strip()

    if not line:
        continue

    # Check if line contains a WhatsApp message
    if " - " not in line:
        system_messages += 1
        continue

    try:
        timestamp, remaining = line.split(" - ", 1)

        if ":" not in remaining:
            system_messages += 1
            continue

        sender, text = remaining.split(":", 1)

        sender = sender.strip()
        text = text.strip()

        if text == "<Media omitted>":
            media_messages += 1

        if text == "This message was deleted":
            deleted_messages += 1

        message = {
            "timestamp": timestamp,
            "sender": sender,
            "text": text
        }

        messages.append(message)

    except:
        system_messages += 1

print("="*60)
print("CHAT PARSER SUMMARY")
print("="*60)

print("Total Parsed Messages :", len(messages))
print("System Messages       :", system_messages)
print("Media Messages        :", media_messages)
print("Deleted Messages      :", deleted_messages)

CHAT PARSER SUMMARY
Total Parsed Messages : 3174
System Messages       : 4
Media Messages        : 32
Deleted Messages      : 15


In [ ]:
print("First 5 Parsed Messages:\n")

for msg in messages[:5]:
    print(msg)

First 5 Parsed Messages:

{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abey'}


# Feature 3 – Group Overview

In [ ]:
# ==========================================================
# Feature 3 : Group Overview
# ==========================================================

participants = {}
dates = []

for msg in messages:

    sender = msg["sender"]

    if sender in participants:
        participants[sender] += 1
    else:
        participants[sender] = 1

    dates.append(msg["timestamp"].split(",")[0])

print("="*60)
print("GROUP OVERVIEW")
print("="*60)

print("Total Messages     :", len(messages))
print("Total Participants :", len(participants))
print("Date Range         :", min(dates), "to", max(dates))

print("\nMessage Count Per Person\n")

sorted_people = sorted(participants.items(), key=lambda x: x[1], reverse=True)

for person, count in sorted_people:
    percentage = (count / len(messages)) * 100
    print(f"{person:10} : {count:5} messages ({percentage:.2f}%)")

GROUP OVERVIEW
Total Messages     : 3174
Total Participants : 6
Date Range         : 01/04/24 to 30/05/24

Message Count Per Person

Rahul      :   953 messages (30.03%)
Priya      :   718 messages (22.62%)
Neha       :   635 messages (20.01%)
Aman       :   490 messages (15.44%)
Karan      :   354 messages (11.15%)
Vikas      :    24 messages (0.76%)


## 📊 Feature 4 – Most Active Day & Hour

This section analyzes chat activity to determine the busiest day and the busiest hour of the group conversation.

In [ ]:
# ==========================================================
# Feature 4 : Most Active Day & Hour
# ==========================================================

day_count = {}
hour_count = {}

for msg in messages:

    timestamp = msg["timestamp"]

    date, time = timestamp.split(",")

    hour = int(time.strip().split(":")[0])

    # Count messages per day
    if date in day_count:
        day_count[date] += 1
    else:
        day_count[date] = 1

    # Count messages per hour
    if hour in hour_count:
        hour_count[hour] += 1
    else:
        hour_count[hour] = 1

# Find busiest day
most_active_day = max(day_count, key=day_count.get)

# Find busiest hour
most_active_hour = max(hour_count, key=hour_count.get)

print("=" * 60)
print("MOST ACTIVE DAY & HOUR")
print("=" * 60)

print(f"Most Active Day  : {most_active_day} ({day_count[most_active_day]} messages)")
print(f"Most Active Hour : {most_active_hour}:00 - {most_active_hour}:59 ({hour_count[most_active_hour]} messages)")

MOST ACTIVE DAY & HOUR
Most Active Day  : 04/05/24 (76 messages)
Most Active Hour : 18:00 - 18:59 (248 messages)


# 📊 Feature 5 – NumPy Activity Heatmap

This feature visualizes the hourly messaging activity of every participant using a NumPy matrix.

The heatmap helps identify:

- Peak chatting hours
- Individual activity patterns
- Group engagement trends
- Most active time period of the day

In [ ]:
import numpy as np

# ==========================================================
# FEATURE 5 : NUMPY ACTIVITY HEATMAP
# ==========================================================

# Create participant list
participants = sorted(list(set(msg["sender"] for msg in messages)))

participant_index = {}

for i, person in enumerate(participants):
    participant_index[person] = i

# NumPy matrix
heatmap = np.zeros((len(participants),24),dtype=int)

# Fill matrix
for msg in messages:

    sender = msg["sender"]

    hour = int(msg["timestamp"].split(",")[1].strip().split(":")[0])

    row = participant_index[sender]

    heatmap[row][hour] += 1


print("="*95)
print("📊 GROUP ACTIVITY HEATMAP")
print("="*95)

print(f"{'Participant':<12}", end="")

for h in range(24):
    print(f"{h:>4}", end="")

print()

print("-"*95)

for i, person in enumerate(participants):

    print(f"{person:<12}", end="")

    for h in range(24):
        print(f"{heatmap[i][h]:>4}", end="")

    print()

📊 GROUP ACTIVITY HEATMAP
Participant    0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23
-----------------------------------------------------------------------------------------------
Aman          54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7  16   8  13  11   0  56
Karan          0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27  25  32  23  14   9   8
Neha           0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30
Priya          0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18   9
Rahul          3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54
Vikas          0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2


In [ ]:
print("\n")
print("="*70)
print("⏰ PEAK ACTIVITY SUMMARY")
print("="*70)

for i, person in enumerate(participants):

    busiest_hour = np.argmax(heatmap[i])

    messages_sent = heatmap[i][busiest_hour]

    if busiest_hour < 6:
        period = "🌙 Night"

    elif busiest_hour < 12:
        period = "🌅 Morning"

    elif busiest_hour < 17:
        period = "☀️ Afternoon"

    else:
        period = "🌆 Evening"

    print(f"{person:<10}")
    print(f"   Peak Hour      : {busiest_hour}:00")
    print(f"   Messages       : {messages_sent}")
    print(f"   Active Period  : {period}")
    print("-"*45)



⏰ PEAK ACTIVITY SUMMARY
Aman      
   Peak Hour      : 4:00
   Messages       : 88
   Active Period  : 🌙 Night
---------------------------------------------
Karan     
   Peak Hour      : 12:00
   Messages       : 37
   Active Period  : ☀️ Afternoon
---------------------------------------------
Neha      
   Peak Hour      : 18:00
   Messages       : 62
   Active Period  : 🌆 Evening
---------------------------------------------
Priya     
   Peak Hour      : 9:00
   Messages       : 65
   Active Period  : 🌅 Morning
---------------------------------------------
Rahul     
   Peak Hour      : 18:00
   Messages       : 105
   Active Period  : 🌆 Evening
---------------------------------------------
Vikas     
   Peak Hour      : 8:00
   Messages       : 3
   Active Period  : 🌅 Morning
---------------------------------------------


In [ ]:
print("\n")
print("="*70)
print("🏆 HOURLY CHAMPIONS")
print("="*70)

for hour in range(24):

    column = heatmap[:,hour]

    winner = np.argmax(column)

    person = participants[winner]

    count = column[winner]

    print(f"{hour:02}:00  →  {person:<10} ({count} messages)")



🏆 HOURLY CHAMPIONS
00:00  →  Aman       (54 messages)
01:00  →  Aman       (67 messages)
02:00  →  Aman       (66 messages)
03:00  →  Aman       (60 messages)
04:00  →  Aman       (88 messages)
05:00  →  Neha       (19 messages)
06:00  →  Rahul      (17 messages)
07:00  →  Priya      (20 messages)
08:00  →  Priya      (47 messages)
09:00  →  Priya      (65 messages)
10:00  →  Priya      (62 messages)
11:00  →  Priya      (61 messages)
12:00  →  Rahul      (58 messages)
13:00  →  Priya      (48 messages)
14:00  →  Rahul      (45 messages)
15:00  →  Rahul      (53 messages)
16:00  →  Rahul      (73 messages)
17:00  →  Rahul      (49 messages)
18:00  →  Rahul      (105 messages)
19:00  →  Rahul      (76 messages)
20:00  →  Neha       (45 messages)
21:00  →  Rahul      (92 messages)
22:00  →  Rahul      (60 messages)
23:00  →  Aman       (56 messages)


In [ ]:
print("\n")
print("="*70)
print("💡 HEATMAP INSIGHTS")
print("="*70)

# Total messages per hour
hour_totals = np.sum(heatmap, axis=0)

overall_peak = np.argmax(hour_totals)

overall_messages = hour_totals[overall_peak]

print(f"📈 Peak group activity occurs around {overall_peak}:00 with {overall_messages} messages.")

quiet = np.argmin(hour_totals)

print(f"😴 Quietest hour is {quiet}:00.")

print()

for i, person in enumerate(participants):

    peak = np.argmax(heatmap[i])

    if peak < 6:
        remark = "prefers late-night conversations 🌙"

    elif peak < 12:
        remark = "is an early bird 🌅"

    elif peak < 17:
        remark = "is active during afternoons ☀️"

    else:
        remark = "comes alive in the evening 🌆"

    print(f"• {person} {remark}.")



💡 HEATMAP INSIGHTS
📈 Peak group activity occurs around 18:00 with 248 messages.
😴 Quietest hour is 5:00.

• Aman prefers late-night conversations 🌙.
• Karan is active during afternoons ☀️.
• Neha comes alive in the evening 🌆.
• Priya is an early bird 🌅.
• Rahul comes alive in the evening 🌆.
• Vikas is an early bird 🌅.


In [ ]:
print("\n")
print("="*70)
print("🎯 GROUP PRIME TIME")
print("="*70)

print(f"The best time to text this group is around {overall_peak}:00.")
print("That's when most members are active and conversations are most lively! 🚀")



🎯 GROUP PRIME TIME
The best time to text this group is around 18:00.
That's when most members are active and conversations are most lively! 🚀


# Feature 6 – Top 10 Most Used Words

In [ ]:
# ==========================================================
# Feature 6 : Top 10 Most Used Words
# ==========================================================

stop_words = {
    "the","is","a","an","to","of","in","on","and","for",
    "i","you","me","my","we","our","it","this","that",
    "are","was","am","be","have","has","had","will",
    "at","with","as","but","or","so","if","im","its",
    "he","she","they","them","his","her","their",
    "do","does","did","can","could","would","should",
    "how","what","when","where","why","who","which",
    "from","about","just","today","tomorrow","yesterday",
    "everyone","anyone","someone","anything","everything",
    "okay","ok","yeah","yes","no","nah","haha","lol",
    "hai","haan","yaar","bhai","abey","na","tu","kya",
    "please","guys"
}

word_count = {}

for msg in messages:

    text = msg["text"].lower()

    # Remove punctuation
    for ch in ".,!?;:()[]{}'\"-_<>/\n":
        text = text.replace(ch, " ")

    words = text.split()

    for word in words:

        if word in stop_words:
            continue

        if word.isdigit():
            continue

        if len(word) <= 1:
            continue

        if word in word_count:
            word_count[word] += 1
        else:
            word_count[word] = 1

# Sort in descending order
top_words = sorted(word_count.items(), key=lambda x: x[1], reverse=True)

print("=" * 60)
print("TOP 10 MOST USED WORDS")
print("=" * 60)

for word, count in top_words[:10]:
    bar = "█" * min(count // 5, 30)
    print(f"{word:<15} {count:>5} {bar}")

TOP 10 MOST USED WORDS
telling           179 ██████████████████████████████
up                172 ██████████████████████████████
one               157 ██████████████████████████████
started           150 ██████████████████████████████
scene             145 █████████████████████████████
entire            145 █████████████████████████████
now               121 ████████████████████████
came              116 ███████████████████████
been              112 ██████████████████████
sleep             112 ██████████████████████


# Feature 7 – Response Analysis

In [ ]:
# ==========================================================
# Feature 7 : Response Analysis
# ==========================================================

from datetime import datetime

response_times = {}

longest_gap = 0
gap_start = ""
gap_end = ""

previous = None

for msg in messages:

    current_time = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")

    if previous is not None:

        previous_time = datetime.strptime(previous["timestamp"], "%d/%m/%y, %H:%M")

        gap = (current_time - previous_time).total_seconds() / 60

        # Longest silent gap
        if gap > longest_gap:
            longest_gap = gap
            gap_start = previous["timestamp"]
            gap_end = msg["timestamp"]

        # Response time only if different sender
        if msg["sender"] != previous["sender"]:

            sender = msg["sender"]

            if sender not in response_times:
                response_times[sender] = []

            response_times[sender].append(gap)

    previous = msg

# Average response time
average_response = {}

for sender in response_times:

    average_response[sender] = sum(response_times[sender]) / len(response_times[sender])

fastest = min(average_response, key=average_response.get)
slowest = max(average_response, key=average_response.get)

print("="*60)
print("RESPONSE ANALYSIS")
print("="*60)

print(f"Fastest Replier : {fastest}")
print(f"Average Response : {average_response[fastest]:.2f} minutes\n")

print(f"Slowest Replier : {slowest}")
print(f"Average Response : {average_response[slowest]:.2f} minutes\n")

print(f"Longest Silent Streak : {longest_gap:.2f} minutes")
print(f"From : {gap_start}")
print(f"To   : {gap_end}")

RESPONSE ANALYSIS
Fastest Replier : Rahul
Average Response : 34.95 minutes

Slowest Replier : Aman
Average Response : 55.36 minutes

Longest Silent Streak : 359.00 minutes
From : 27/04/24, 04:30
To   : 27/04/24, 10:29


# Feature 8 – Personality Detection

In [ ]:
# ==========================================================
# Feature 8 : Personality Detection
# ==========================================================

message_count = {}
night_count = {}
long_message_count = {}
emotion_count = {}

for person in participants:
    message_count[person] = 0
    night_count[person] = 0
    long_message_count[person] = 0
    emotion_count[person] = 0

for msg in messages:

    sender = msg["sender"]
    text = msg["text"]

    message_count[sender] += 1

    hour = int(msg["timestamp"].split(",")[1].strip().split(":")[0])

    if 0 <= hour <= 5:
        night_count[sender] += 1

    if len(text) > 80:
        long_message_count[sender] += 1

    if "!" in text or text.isupper():
        emotion_count[sender] += 1

spammer = max(message_count, key=message_count.get)
night_owl = max(night_count, key=night_count.get)
story_teller = max(long_message_count, key=long_message_count.get)
drama_queen = max(emotion_count, key=emotion_count.get)
ghost = min(message_count, key=message_count.get)

print("="*65)
print("🧠 PERSONALITY ANALYSIS")
print("="*65)

print(f"🔥 Spammer       : {spammer}")
print(f"🌙 Night Owl     : {night_owl}")
print(f"📖 Story Teller  : {story_teller}")
print(f"🎭 Drama Queen   : {drama_queen}")
print(f"👻 Ghost         : {ghost}")

🧠 PERSONALITY ANALYSIS
🔥 Spammer       : Rahul
🌙 Night Owl     : Aman
📖 Story Teller  : Karan
🎭 Drama Queen   : Neha
👻 Ghost         : Vikas


# Feature 9 – Conversation Starter Analysis

In [ ]:
print("="*70)
print("🗣 CONVERSATION STARTER ANALYSIS")
print("="*70)

conversation_starters = {}
started_dates = set()

for person in participants:
    conversation_starters[person] = 0

for msg in messages:
    date = msg["timestamp"].split(",")[0].strip()

    if date not in started_dates:
        conversation_starters[msg["sender"]] += 1
        started_dates.add(date)

for person, count in sorted(conversation_starters.items(), key=lambda x: x[1], reverse=True):
    print(f"{person:<10} started {count} conversation(s)")

starter = max(conversation_starters, key=conversation_starters.get)

print("\n🏆 Conversation Starter Award :", starter)

🗣 CONVERSATION STARTER ANALYSIS
Aman       started 58 conversation(s)
Rahul      started 2 conversation(s)
Karan      started 0 conversation(s)
Neha       started 0 conversation(s)
Priya      started 0 conversation(s)
Vikas      started 0 conversation(s)

🏆 Conversation Starter Award : Aman


# Feature 10 – Communication Style

In [ ]:
print("="*70)
print("📝 COMMUNICATION STYLE")
print("="*70)

total_length = {}
message_number = {}

for person in participants:
    total_length[person] = 0
    message_number[person] = 0

for msg in messages:
    sender = msg["sender"]
    total_length[sender] += len(msg["text"])
    message_number[sender] += 1

avg_length = {}

for person in participants:

    avg = total_length[person] / message_number[person]
    avg_length[person] = avg

    if avg < 20:
        style = "Short & Quick"
    elif avg < 50:
        style = "Balanced Communicator"
    else:
        style = "Detailed Story Teller"

    print(f"{person:<10} {avg:.2f} characters/message   --> {style}")

📝 COMMUNICATION STYLE
Aman       26.62 characters/message   --> Balanced Communicator
Karan      303.93 characters/message   --> Detailed Story Teller
Neha       25.23 characters/message   --> Balanced Communicator
Priya      28.12 characters/message   --> Balanced Communicator
Rahul      10.80 characters/message   --> Short & Quick
Vikas      7.88 characters/message   --> Short & Quick


# Feature 11 – Group Awards

In [ ]:
print("="*70)
print("🏆 GROUP AWARDS")
print("="*70)

print(f"🥇 Chatterbox Award      : {spammer}")
print(f"🌙 Midnight Legend       : {night_owl}")
print(f"📖 Essay Writer          : {story_teller}")
print(f"🎭 Drama Star            : {drama_queen}")
print(f"👻 Silent Observer       : {ghost}")
print(f"🗣 Conversation Starter  : {starter}")

🏆 GROUP AWARDS
🥇 Chatterbox Award      : Rahul
🌙 Midnight Legend       : Aman
📖 Essay Writer          : Karan
🎭 Drama Star            : Neha
👻 Silent Observer       : Vikas
🗣 Conversation Starter  : Aman


# Feature 12 – AI Insights

In [ ]:
print("="*70)
print("🤖 GROUP INSIGHTS")
print("="*70)

most_active = sorted_people[0][0]
least_active = sorted_people[-1][0]

print(f"• {most_active} is the most active member of the group.")
print(f"• {least_active} participates the least in conversations.")
print(f"• The group is most active on {most_active_day}.")
print(f"• Peak chatting happens around {most_active_hour}:00.")
print(f"• {night_owl} prefers chatting during late-night hours.")
print(f"• {story_teller} usually sends longer and more detailed messages.")
print(f"• {starter} most frequently starts the day's conversations.")
print("• Overall, the group shows healthy interaction among multiple members.")

🤖 GROUP INSIGHTS
• Rahul is the most active member of the group.
• Vikas participates the least in conversations.
• The group is most active on 04/05/24.
• Peak chatting happens around 18:00.
• Aman prefers chatting during late-night hours.
• Karan usually sends longer and more detailed messages.
• Aman most frequently starts the day's conversations.
• Overall, the group shows healthy interaction among multiple members.


# Final GroupDNA Report

In [ ]:
print("\n" + "═"*75)
print("📱                 GROUPDNA ANALYTICS DASHBOARD")
print("═"*75)
print("        Turning Conversations into Meaningful Insights")
print("═"*75)

print("\n📊 GROUP SNAPSHOT")
print("─"*75)
print(f"👥 Participants        : {len(participants)}")
print(f"💬 Total Messages      : {len(messages)}")
print(f"📅 Chat Duration       : {min(dates)}  ➜  {max(dates)}")
print(f"📈 Most Active Day     : {most_active_day}")
print(f"⏰ Peak Chat Hour      : {most_active_hour}:00")

print("\n🏆 HALL OF FAME")
print("─"*75)
print(f"🔥 Chatterbox Champion     : {spammer}")
print(f"🌙 Midnight Philosopher    : {night_owl}")
print(f"📖 Storytelling Expert     : {story_teller}")
print(f"🎭 Drama Department Head   : {drama_queen}")
print(f"👻 Silent Observer         : {ghost}")
print(f"🗣 Conversation Starter    : {starter}")

print("\n📚 COMMUNICATION STATS")
print("─"*75)

for person, avg in sorted(avg_length.items(), key=lambda x: x[1], reverse=True):

    if avg > 60:
        badge = "📝 Essay Mode"
    elif avg > 30:
        badge = "💬 Balanced"
    else:
        badge = "⚡ Quick Texter"

    print(f"{person:<10} {avg:6.2f} chars/msg   {badge}")

print("\n💡 GROUP INSIGHTS")
print("─"*75)

print(f"✅ {spammer} definitely keeps this group alive.")
print(f"🌙 Don't expect {night_owl} to reply before sunrise.")
print(f"📖 If there's a long message, it's probably from {story_teller}.")
print(f"👻 {ghost} has mastered the art of 'seen... maybe later.'")
print(f"🗣 {starter} is usually the one who breaks the silence.")
print(f"📈 The group comes alive around {most_active_hour}:00.")

print("\n📈 OVERALL GROUP HEALTH")
print("─"*75)

participation = len([p for p,c in sorted_people if c > 50])

if participation >= 5:
    health = "⭐⭐⭐⭐⭐  Excellent"
elif participation >= 4:
    health = "⭐⭐⭐⭐☆  Very Good"
elif participation >= 3:
    health = "⭐⭐⭐☆☆  Good"
else:
    health = "⭐⭐☆☆☆  Needs More Interaction"

print("Community Engagement :", health)

print("\n🎉 FINAL VERDICT")
print("─"*75)

print("Every group has a unique personality.")
print("This conversation shows collaboration, regular activity,")
print("and a healthy mix of quick chats, long discussions,")
print("late-night thoughts, and silent observers.")
print("In short... this group definitely has character! 😄")

print("\n" + "═"*75)
print("✅ Analysis Completed Successfully")
print("      Thank you for exploring GroupDNA!")
print("═"*75)


═══════════════════════════════════════════════════════════════════════════
📱                 GROUPDNA ANALYTICS DASHBOARD
═══════════════════════════════════════════════════════════════════════════
        Turning Conversations into Meaningful Insights
═══════════════════════════════════════════════════════════════════════════

📊 GROUP SNAPSHOT
───────────────────────────────────────────────────────────────────────────
👥 Participants        : 6
💬 Total Messages      : 3174
📅 Chat Duration       : 01/04/24  ➜  30/05/24
📈 Most Active Day     : 04/05/24
⏰ Peak Chat Hour      : 18:00

🏆 HALL OF FAME
───────────────────────────────────────────────────────────────────────────
🔥 Chatterbox Champion     : Rahul
🌙 Midnight Philosopher    : Aman
📖 Storytelling Expert     : Karan
🎭 Drama Department Head   : Neha
👻 Silent Observer         : Vikas
🗣 Conversation Starter    : Aman

📚 COMMUNICATION STATS
───────────────────────────────────────────────────────────────────────────
Karan      303.93 c

# Conclusion

This project successfully analyzes a WhatsApp group chat using Python fundamentals and NumPy.

The analyzer extracts messages, computes participant statistics, identifies activity patterns, generates a NumPy-based activity heatmap, analyzes commonly used words, evaluates response behavior, and assigns personality archetypes based on communication patterns.

The implementation follows the project constraints by avoiding Pandas and using only Python fundamentals, NumPy, and datetime.